In [4]:
import numpy as np

# Set the seed for reproducibility
np.random.seed(5)

n = 10

# Generate mean of the portfolio pbar
pbar = np.ones((n, 1)) * 0.03 + np.vstack((np.random.rand(n-1, 1), np.array([[0]]))) * 0.12

# Generate covariance matrix of the portfolio Sigma
Sigma = np.random.randn(n, n)
Sigma = np.dot(Sigma.T, Sigma)
Sigma = Sigma / np.max(np.abs(np.diag(Sigma))) * 0.2
Sigma[:, n-1] = np.zeros(n)
Sigma[n-1, :] = np.zeros((1, n))

# If you want to verify the positive semidefiniteness of the matrix Sigma
# Calculate the eigenvalues of S
eigenvalues = np.linalg.eigvals(Sigma)

# Check if all eigenvalues are non-negative
is_positive_semidefinite = np.all(eigenvalues >= 0)
print("Is Sigma positive semidefinite?", is_positive_semidefinite)



Is Sigma positive semidefinite? True


In [2]:
import enum
import pyomo.environ as pyo 

SOLVER = pyo.SolverFactory('gurobi')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)


Solver ready: <pyomo.solvers.plugins.solvers.GUROBI.GUROBIFILE object at 0x000002CA7A0A9400>


In [11]:
model = pyo.ConcreteModel()

model.n = pyo.Set(initialize=range(n))
model.x_long = pyo.Var(range(n), domain=pyo.NonNegativeReals)
model.x_short = pyo.Var(range(n), domain=pyo.NonNegativeReals)

def objective(model):
    return sum(
        (model.x_long[i] - model.x_short[i]) * Sigma[i, j] * (model.x_long[j] - model.x_short[j])
        for i in model.n
        for j in model.n
    )
    
    
model.objective = pyo.Objective(rule=objective, sense=pyo.minimize)


def pbar_constraint(model):
    return sum((model.x_long[i] - model.x_short[i]) * pbar[i] for i in model.n) >= .1

model.pbar_constraint = pyo.Constraint(rule=pbar_constraint)

def budget_constraint(model):
    return sum(model.x_long[i] - model.x_short[i] for i in model.n) == 1
model.budget_constraint = pyo.Constraint(rule=budget_constraint)

def total_short(model):
    return sum(model.x_short[i] for i in model.n) <= 0.5
model.total_short = pyo.Constraint(rule=total_short)



SOLVER.solve(model)
x_long_values = [pyo.value(model.x_long[i]) for i in model.n]
x_short_values = [pyo.value(model.x_short[i]) for i in model.n]
print("Long positions:", np.round(x_long_values, 5))
print("Short positions:", np.round(x_short_values, 5))


Long positions: [0.0093  0.26697 0.0068  0.22213 0.14796 0.1878  0.20349 0.00695 0.00471
 0.43667]
Short positions: [0.04668 0.00352 0.05072 0.00325 0.00405 0.00385 0.00376 0.27023 0.10262
 0.0041 ]
